In [3]:
import pandas as pd
import numpy as np
import os
import sys

In [4]:
import Bio
from Bio.Seq import Seq
from Bio import SeqIO
from Bio.Restriction import AllEnzymes

In [5]:
plasmid_dict = {
    "pGEX-4T-1": {
        "RE_site": ["BamHI", "EagI"],
        "file_path": "./input/plasmids/pGEX-4T-1.fa"
    },
    "pMAL-c5X": {
        "RE_site": ["NdeI", "HindIII"],
        "file_path": "./input/plasmids/pMAL-c5X.fa"
    },
    "pET-21a(+)": {
        "RE_site": ["NdeI", "XhoI"],
        "file_path": "./input/plasmids/pET-21a(+).fa"
    },
    "pET-28a(+)": {
        "RE_site": ["BamHI", "XhoI"],
        "file_path": "./input/plasmids/pET-28a(+).fa"
    },
    "pET-28a(+)_start_codon": {
        "RE_site": ["NcoI", "XhoI"],
        "file_path": "./input/plasmids/pET-28a(+).fa"
    },
    "pCold_I": {
        "RE_site": ["NdeI", "BspMI"],
        "file_path": "./input/plasmids/pCold_I.fa"
    },
    "pUC18": {
        "RE_site": ["EcoRI", "HindIII"],
        "file_path": "./input/plasmids/pUC18.fa"
    },
    "pQE-3": {
        "RE_site": ["BamHI", "HindIII"],
        "file_path": "./input/plasmids/pQE-3.fa"
    }
}



In [6]:
df_plasmid = pd.DataFrame.from_dict(plasmid_dict, orient='index')
#df_plasmid.reset_index(inplace=True)
#df_plasmid.rename(columns={'index': 'Plasmid'}, inplace=True)
df_plasmid['Plasmid'] = df_plasmid.index
df_plasmid['insert_site_1'] = df_plasmid['RE_site'].apply(lambda x: sorted(x)[0])
df_plasmid['insert_site_2'] = df_plasmid['RE_site'].apply(lambda x: sorted(x)[1])
df_plasmid

,RE_site,file_path,Plasmid,insert_site_1,insert_site_2
pGEX-4T-1,"[BamHI, EagI]",./input/plasmids/pGEX-4T-1.fa,pGEX-4T-1,BamHI,EagI
pMAL-c5X,"[NdeI, HindIII]",./input/plasmids/pMAL-c5X.fa,pMAL-c5X,HindIII,NdeI
pET-21a(+),"[NdeI, XhoI]",./input/plasmids/pET-21a(+).fa,pET-21a(+),NdeI,XhoI
pET-28a(+),"[BamHI, XhoI]",./input/plasmids/pET-28a(+).fa,pET-28a(+),BamHI,XhoI
pET-28a(+)_start_codon,"[NcoI, XhoI]",./input/plasmids/pET-28a(+).fa,pET-28a(+)_start_codon,NcoI,XhoI
pCold_I,"[NdeI, BspMI]",./input/plasmids/pCold_I.fa,pCold_I,BspMI,NdeI
pUC18,"[EcoRI, HindIII]",./input/plasmids/pUC18.fa,pUC18,EcoRI,HindIII
pQE-3,"[BamHI, HindIII]",./input/plasmids/pQE-3.fa,pQE-3,BamHI,HindIII


In [7]:
def can_digest(dna_seq, enzyme_name):
    """
    Check if a given restriction enzyme can digest the DNA sequence.

    Args:
        dna_seq (str): The DNA sequence.
        enzyme_name (str): The name of the restriction enzyme (must be defined in Bio.Restriction).

    Returns:
        bool: True if the enzyme can cut the sequence, False otherwise.
    """
    try:
        # Get the enzyme from Bio.Restriction using its name
        enzyme = getattr(Bio.Restriction, enzyme_name)
    except AttributeError:
        print(f"Enzyme '{enzyme_name}' not found in Bio.Restriction.")
        return False

    # Search for restriction sites in the given DNA sequence
    sites = enzyme.search(Seq(dna_seq))
    return len(sites) > 0


def read_plasmid_dna(file_path):
    """
    Read a plasmid DNA sequence from a .dna file using Biopython's SeqIO.
    
    Args:
        file_path (str): Path to the .dna file (assumed to be in FASTA format).
        
    Returns:
        Bio.Seq.Seq: The DNA sequence as a Biopython Seq object.
    """
    record = SeqIO.read(file_path, "fasta")
    return record.seq


def digest_plasmid_and_test_enzyme(re_test, plasmid_file='pET-28a(+).fa', re1="NcoI", re2="XhoI", longer_frag=True):
    # Read the plasmid sequence from the file
    plasmid_seq = read_plasmid_dna(plasmid_file)

    # Dynamically get the two enzymes from Bio.Restriction using their names
    try:
        enzyme1 = getattr(Bio.Restriction, re1)
        enzyme2 = getattr(Bio.Restriction, re2)
        enzyme_test = getattr(Bio.Restriction, re_test)
    except AttributeError as e:
        print(f"Error: {e}")
        return

    # Digest the cyclic plasmid with both enzymes (each should cut exactly once)
    cuts1 = enzyme1.search(plasmid_seq, linear=False)
    cuts2 = enzyme2.search(plasmid_seq, linear=False)

    if len(cuts1) != 1 or len(cuts2) != 1:
        print("Each enzyme must cut the plasmid exactly once for proper digestion.")
        return

    pos1, pos2 = sorted(cuts1 + cuts2)
    frag1 = plasmid_seq[pos1:pos2]
    frag2 = plasmid_seq[pos2:] + plasmid_seq[:pos1]

    # Select the fragment based on the user's choice
    if longer_frag:
        selected_frag = frag1 if len(frag1) >= len(frag2) else frag2
    else:  # Select the shorter fragment
        selected_frag = frag1 if len(frag1) < len(frag2) else frag2

    #print(f"Select longer fragment is {longer_frag}, with length {len(selected_frag)}")

    # Test whether the given enzyme can digest the selected fragment
    sites = enzyme_test.search(selected_frag)
    return len(sites) == 0


In [8]:
def plasmid_digest_check(row, enzyme_name):
    
    plasmid_file = row['file_path']
    re1 = row['insert_site_1']
    re2 = row['insert_site_2']
    # Check if the enzyme can digest the plasmid
    result = digest_plasmid_and_test_enzyme(enzyme_name, plasmid_file=plasmid_file, re1=re1, re2=re2, longer_frag=True)
    return result

In [9]:
from Bio.Restriction import AllEnzymes

for enzyme in AllEnzymes:

    enzyme_name = enzyme.__name__
    df_plasmid[enzyme_name] = df_plasmid.apply(lambda row: plasmid_digest_check(row, enzyme_name), axis=1)

In [10]:
df_plasmid = df_plasmid.T

header = ['Plasmid', 'RE_site', 'file_path', 'insert_site_1', 'insert_site_2']

df_plasmid = df_plasmid.drop(header, axis=0)
df_plasmid

,pGEX-4T-1,pMAL-c5X,pET-21a(+),pET-28a(+),pET-28a(+)_start_codon,pCold_I,pUC18,pQE-3
EcoRV,False,True,False,False,False,False,True,True
HindII,False,False,False,False,False,False,True,True
Cfr42I,True,True,True,True,True,True,True,True
AanI,True,False,False,False,False,False,True,False
AquIII,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...
BciVI,False,False,False,False,False,False,False,False
TspDTI,False,False,False,False,False,False,False,False
BarI,True,True,False,False,False,True,True,True
AccIII,True,False,False,False,False,True,True,False


In [11]:
df_plasmid.to_csv('./output/plasmid_check.csv', index=True, header=True)

In [12]:
df_plasmid = pd.read_csv('./output/plasmid_check.csv', index_col=0)

In [13]:
df_plasmid

,pGEX-4T-1,pMAL-c5X,pET-21a(+),pET-28a(+),pET-28a(+)_start_codon,pCold_I,pUC18,pQE-3
EcoRV,False,True,False,False,False,False,True,True
HindII,False,False,False,False,False,False,True,True
Cfr42I,True,True,True,True,True,True,True,True
AanI,True,False,False,False,False,False,True,False
AquIII,True,True,True,True,True,True,True,True
...,...,...,...,...,...,...,...,...
BciVI,False,False,False,False,False,False,False,False
TspDTI,False,False,False,False,False,False,False,False
BarI,True,True,False,False,False,True,True,True
AccIII,True,False,False,False,False,True,True,False
